In [1]:
from pyarrow import dataset as ds
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

### load data

In [2]:
dataset = ds.dataset("../data/renovation-data-all.parquet", format="parquet")
table = dataset.to_table(columns=None, use_threads=True)
df_preprocessed = table.to_pandas()
print(f"shape: {df_preprocessed.shape}")
df_preprocessed.head()

shape: (39736, 83)


,entity_id,epc_before,ontology.building.building_general.construction_year,ontology.building.building_general.floor_area,ontology.building.pv.available_roof_area_for_solar_panel_in_m2,ontology.building.pv.has_pv,steps,has_insulate_exterior_inclined_roof,has_insulate_exterior_flat_roof,has_insulate_exterior_wall,...,ontology.building.systems.space_heating_emission_systems.CONVECTOR,ontology.building.systems.space_heating_emission_systems.FLOOR,ontology.building.systems.space_heating_emission_systems.AIR,ontology.building.systems.space_heating_emission_systems.LOCAL_STOVE,ontology.building.envelope.insulation_level_roof_extension_MODERATELY_INSULATED,ontology.building.envelope.insulation_level_roof_main_MODERATELY_INSULATED,ontology.building.systems.space_heating_generation_on_electricity.CENTRAL_HEAT_PUMP,ontology.building.systems.space_heating_generation_on_electricity.LOCAL_RESISTANCE_HEATERS,ontology.building.envelope.basement_occurance_PARTIAL,ontology.building.envelope.floor_boundary_types.BASEMENT_OR_CRAWLSPACE
0,0,394,1965.0,203.0,8.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,0,0,0,0,0,0,0,1
1,1,413,1969.0,226.0,16.0,0,"[insulate_floors_above_air_from_below, insulat...",1,1,1,...,1,0,1,0,1,0,0,0,0,1
2,2,469,1967.0,195.0,71.0,0,"[insulate_exterior_inclined_roof, replace_sing...",1,1,1,...,0,0,0,0,1,0,0,0,0,1
3,3,592,1969.0,220.0,63.0,0,"[insulate_floors_above_air_from_below, replace...",0,1,0,...,1,1,0,0,0,0,0,0,0,1
4,5,511,1967.0,173.0,25.0,0,"[replace_epump_with_dhw, insulate_floors_above...",1,1,1,...,0,0,0,1,1,1,0,0,0,1


### train model

In [3]:
steps_encoding = {
    'insulate_exterior_inclined_roof': 1,
    'insulate_exterior_flat_roof': 2,
    'insulate_exterior_wall': 3,
    'insulate_cavity_wall': 4,
    'insulate_floors_above_air_from_below': 5,
    'replace_single_and_old_double_for_hr_glass_with_new_frames': 6,
    'replace_non_condensing_gas_boiler_with_condensing_gas_boiler': 7,
    'replace_epump_with_dhw': 8,
    'install_pv_panels': 9,  
}

def get_columns_to_drop(df, component):
    columns_to_drop = ["entity_id"]

    if component == "relevant_steps":
        columns_to_drop.extend([col for col in df.columns if col.startswith("has_")])
        columns_to_drop.extend(["step_count", "steps", "steps_gt"])
    elif component == "first_step":
        columns_to_drop.extend(["first_step", "steps", "steps_gt"]) # remove steps(gt) since list, but same information is in 'has_' columns

    return columns_to_drop

In [4]:
encoding_empty = 0
RFC_params = {'random_state': 42, 'class_weight': 'balanced'}

# define target
df_preprocessed["first_step"] = df_preprocessed["steps"].apply(lambda x: x[0] if len(x)>0 else None)
df_preprocessed["first_step"].value_counts(dropna=False)
y = df_preprocessed["first_step"].apply(lambda x: steps_encoding[x] if x in steps_encoding else encoding_empty)

# define features
columns_to_drop = get_columns_to_drop(df_preprocessed, component="first_step")
X = df_preprocessed.drop(columns=columns_to_drop)
print(f"{X.shape[1]} features used for training: {X.columns.tolist()}")

###################################
##### out-of-fold predictions #####
###################################

# using StratifiedKFold to maintain class distribution in each fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

K = len(steps_encoding) + 1 # number of classes + 1 for encoding_empty
oof_proba = np.zeros((len(y), K))

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_va = y[train_idx], y[valid_idx]

    RFC_model = RandomForestClassifier(**RFC_params)
    RFC_model.fit(X_tr, y_tr)

    oof_proba[valid_idx, :] = RFC_model.predict_proba(X_va)

y_pred = np.argmax(oof_proba, axis=1)
report = classification_report(y, y_pred, output_dict=True)

################################
##### training final model #####
################################

final_model = RandomForestClassifier(**RFC_params)
final_model.fit(X, y)

filename = '../models/first_step_RF.pkl'
with open(filename, 'wb') as file:
    pickle.dump({
        "model_RF": final_model,
        "oof_predictions": oof_proba,
    }, file)

print("Saved:", filename)

80 features used for training: ['epc_before', 'ontology.building.building_general.construction_year', 'ontology.building.building_general.floor_area', 'ontology.building.pv.available_roof_area_for_solar_panel_in_m2', 'ontology.building.pv.has_pv', 'has_insulate_exterior_inclined_roof', 'has_insulate_exterior_flat_roof', 'has_insulate_exterior_wall', 'has_insulate_cavity_wall', 'has_insulate_floors_above_air_from_below', 'has_replace_single_and_old_double_for_hr_glass_with_new_frames', 'has_replace_non_condensing_gas_boiler_with_condensing_gas_boiler', 'has_replace_epump_with_dhw', 'has_install_pv_panels', 'step_count', 'ontology.building.energy_carriers.cooking.GAS_DISTRICT', 'ontology.building.energy_carriers.cooking.ELECTRICITY', 'ontology.building.energy_carriers.space_cooling.ELECTRICITY', 'ontology.building.systems.hot_water_system_types_on_electricity.ELECTRICAL_RESISTANCE', 'ontology.building.systems.hot_water_system_types_on_electricity.HEAT_PUMP', 'ontology.building.energy_car

### analyse results

In [6]:
f1_scores = [report[row]["f1-score"] for row in report if row not in ["accuracy", "macro avg", "weighted avg"]]
supports = [report[row]["support"] for row in report if row not in ["accuracy", "macro avg", "weighted avg"]]

results_df = pd.DataFrame({
    "F1 Score": f1_scores, 
    "Support": supports
    }, index=[None] + list(steps_encoding.keys()))

display(results_df.round(3))
print(f"Weighted average F1 score: {report['weighted avg']['f1-score']:.3f}")

,F1 Score,Support
None,1.000,301.0
insulate_exterior_inclined_roof,0.999,2993.0
insulate_exterior_flat_roof,0.969,418.0
insulate_exterior_wall,0.871,70.0
insulate_cavity_wall,0.833,28.0
insulate_floors_above_air_from_below,0.998,26485.0
replace_single_and_old_double_for_hr_glass_with_new_frames,0.991,740.0
replace_non_condensing_gas_boiler_with_condensing_gas_boiler,0.974,20.0
replace_epump_with_dhw,1.000,5678.0
install_pv_panels,0.979,3003.0


Weighted average F1 score: 0.996
